# Entrenamiento temporal GRU — implementación reproducible

Este notebook reconstruye la configuración final del experimento temporal **Full384 + GRU**.

> **Importante:** no es el notebook histórico 38 ejecutado originalmente en Colab. Se ha reconstruido a partir del código del modelo, el checkpoint y los artefactos conservados del experimento final.


## Entrada temporal

Cada observación utiliza tres características:

```text
feat_p  = p_t
feat_dp = p_t - p_(t-1)
feat_dt = intervalo temporal normalizado
```

La GRU recibe ventanas **causales de 8 observaciones**. No utiliza información futura.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

from src.temporal_gru import TemporalGRU


In [ ]:
SEQ_LEN = 8
HIDDEN_SIZE = 32
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 8


## Dataset temporal

El CSV de entrenamiento debe contener:

```text
event_id,order,feat_p,feat_dp,feat_dt,label
```

Las filas con etiqueta no supervisada se excluyen de la pérdida. Las ventanas incompletas al inicio de cada evento se rellenan repitiendo la primera observación disponible.


In [ ]:
class TemporalDataset(Dataset):
    def __init__(self, csv_path, seq_len=SEQ_LEN):
        df = pd.read_csv(csv_path)
        self.samples = []

        for _, group in df.groupby("event_id"):
            group = group.sort_values("order").reset_index(drop=True)
            feats = group[["feat_p", "feat_dp", "feat_dt"]].to_numpy(dtype="float32")
            labels = group["label"].to_numpy(dtype="float32")

            for i in range(len(group)):
                if np.isnan(labels[i]):
                    continue

                start = max(0, i - seq_len + 1)
                window = feats[start:i + 1]

                if len(window) < seq_len:
                    pad = np.repeat(window[:1], seq_len - len(window), axis=0)
                    window = np.vstack([pad, window])

                self.samples.append((window, labels[i]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)


In [ ]:
def evaluate_auc(model, loader, device):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            ps.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            ys.extend(y.numpy().tolist())

    return roc_auc_score(ys, ps) if len(set(ys)) > 1 else float("nan")


In [ ]:
def train_gru(train_csv, val_csv, output_path, pos_weight=1.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_ds = TemporalDataset(train_csv)
    val_ds = TemporalDataset(val_csv)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = TemporalGRU(
        input_size=3,
        hidden_size=HIDDEN_SIZE,
        num_layers=1,
        dropout=0.2,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight, device=device)
    )

    best_auc = -1.0
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

        val_auc = evaluate_auc(model, val_loader, device)
        print(f"epoch={epoch:03d} val_auc={val_auc:.6f}")

        if val_auc > best_auc:
            best_auc = val_auc
            bad_epochs = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_auc": val_auc,
                    "config": {
                        "seq_len": SEQ_LEN,
                        "features": ["feat_p", "feat_dp", "feat_dt"],
                    },
                },
                output_path,
            )
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break


## Historial real conservado del experimento 38

El mejor checkpoint se obtuvo en la **época 49**, con:

```text
validation AUC = 0.862844
```

El umbral operativo seleccionado sobre validación fue **0.85**.


## Evaluación final sobre 50 eventos de test

| Métrica | Media móvil 5 | GRU |
|---|---:|---:|
| Detección post-t=0 | 0.88 | **0.96** |
| ≤ 5 min | 0.36 | **0.56** |
| ≤ 10 min | 0.58 | **0.72** |
| ≤ 15 min | 0.72 | **0.90** |
| ≤ 30 min | 0.88 | **0.94** |
| Clear-pre alert | **0.18** | 0.28 |
| Mediana hasta primera alerta | 361 s | **210.5 s** |
| Eventos no detectados | 6 | **2** |

La GRU mejora cobertura y rapidez de detección, pero aumenta la sensibilidad antes de la referencia del evento. Por tanto, **no se interpreta como una reducción general de falsos positivos**.


## Nota sobre `feat_dt`

El checkpoint original conserva el nombre de esta característica, pero no serializa el factor exacto usado para normalizar el intervalo temporal. Por ello, una reproducción bit a bit requiere recuperar el mismo preprocesado temporal original. Esta limitación se mantiene explícita en el repositorio.
